In [16]:
import os
import glob
import gzip
from pathlib import Path

import pandas as pd
import scanpy as sc
import scirpy as ir
import muon as mu

In [17]:
mdata = mu.read('GSE139555_Tcells.h5mu')
mdata

e:\Anaconda\envs\bio\Lib\site-packages\mudata\_core\mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
e:\Anaconda\envs\bio\Lib\site-packages\mudata\_core\mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


MuData object with n_obs × n_vars = 111062 × 30727
  obs:	'isT', 'ident', 'patient', 'source', 'type', 'subtype'
  2 modalities
    gex:	111062 x 30727
      obs:	'sample'
    airr:	111062 x 0
      obs:	'sample'
      obsm:	'airr'

In [18]:
# Load T cell metadata and merge with mdata
print("Loading T cell metadata...")
tcell_metadata = pd.read_csv("GSE139555_tcell_metadata.txt", sep="\t", index_col=0)
tcell_metadata


Loading T cell metadata...


,UMAP_1,UMAP_2,ident,patient,sample,source,clonotype
LT1_AAACCTGAGGATATAC-1,0.761298,1.301195,8.3a-Trm,Lung1,LT1,Tumor,lung1.tn.C1
LT1_AAACCTGAGTTACCCA-1,-6.475698,0.690571,4.3-TCF7,Lung1,LT1,Tumor,lung1.tn.C3
LT1_AAACCTGCAACACCCG-1,-1.326519,0.730108,4.4-FOS,Lung1,LT1,Tumor,lung1.tn.C5
LT1_AAACCTGCATCTCCCA-1,-3.510637,1.008227,4.4-FOS,Lung1,LT1,Tumor,lung1.tn.C8
LT1_AAACCTGGTTCGTCTC-1,-5.588617,1.632665,4.3-TCF7,Lung1,LT1,Tumor,lung1.tn.C12
...,...,...,...,...,...,...,...
RB3_TTTGTCACACAGGCCT-1,1.650373,3.015730,8.2-Tem,Renal3,RB3,Blood,renal3.tnb.C38
RB3_TTTGTCACACCCATTC-1,4.788950,-0.804241,8.3b-Trm,Renal3,RB3,Blood,NaN
RB3_TTTGTCACACGCTTTC-1,-1.538078,0.933320,4.4-FOS,Renal3,RB3,Blood,NaN
RB3_TTTGTCATCACGACTA-1,4.730519,-0.530475,8.3b-Trm,Renal3,RB3,Blood,renal3.tnb.C198


In [19]:
# Keep only 'ident' and 'sample' columns
tcell_metadata = tcell_metadata[['ident', 'patient', 'source', 'clonotype']]
# Convert to strings for HDF5 compatibility
for col in tcell_metadata.columns:
    tcell_metadata[col] = tcell_metadata[col].astype(str)
tcell_metadata.head()


C:\Users\a4945\AppData\Local\Temp\ipykernel_27040\3098996309.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tcell_metadata[col] = tcell_metadata[col].astype(str)
C:\Users\a4945\AppData\Local\Temp\ipykernel_27040\3098996309.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tcell_metadata[col] = tcell_metadata[col].astype(str)
C:\Users\a4945\AppData\Local\Temp\ipykernel_27040\3098996309.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_

,ident,patient,source,clonotype
LT1_AAACCTGAGGATATAC-1,8.3a-Trm,Lung1,Tumor,lung1.tn.C1
LT1_AAACCTGAGTTACCCA-1,4.3-TCF7,Lung1,Tumor,lung1.tn.C3
LT1_AAACCTGCAACACCCG-1,4.4-FOS,Lung1,Tumor,lung1.tn.C5
LT1_AAACCTGCATCTCCCA-1,4.4-FOS,Lung1,Tumor,lung1.tn.C8
LT1_AAACCTGGTTCGTCTC-1,4.3-TCF7,Lung1,Tumor,lung1.tn.C12


In [20]:
# Transform mdata index to match tcell_metadata format
# From: 'AAACCTGAGGATATAC-1-GSM4143655_SAM24345862-lt1'
# To: 'LT1_AAACCTGAGGATATAC-1'

print("Before transformation:")
print(f"mdata.obs.index sample (first 3): {list(mdata.obs.index[:3])}")

def transform_barcode(barcode):
    """Transform barcode from 'AAACCTGAGGATATAC-1-GSM4143655_SAM24345862-lt1' to 'LT1_AAACCTGAGGATATAC-1'"""
    # Split by '-GSM' to separate barcode and sample info
    if '-GSM' in barcode:
        parts = barcode.split('-GSM', 1)
        barcode_part = parts[0]  # e.g., 'AAACCTGAGGATATAC-1'
        sample_part = 'GSM' + parts[1]  # e.g., 'GSM4143655_SAM24345862-lt1'
        
        # Extract tissue code (last part after last '-', uppercase it)
        tissue_code = sample_part.rsplit('-', 1)[-1].upper()  # e.g., 'lt1' -> 'LT1'
        
        # Combine: tissue_code + '_' + barcode
        return f"{tissue_code}_{barcode_part}"  # e.g., 'LT1_AAACCTGAGGATATAC-1'
    return barcode

# Transform indices for each modality separately (they have different lengths)
new_gex_index = [transform_barcode(bc) for bc in mdata.mod['gex'].obs_names]
new_airr_index = [transform_barcode(bc) for bc in mdata.mod['airr'].obs_names]
new_mdata_index = [transform_barcode(bc) for bc in mdata.obs_names]

# Update obs names for all modalities
mdata.mod['gex'].obs_names = new_gex_index
mdata.mod['airr'].obs_names = new_airr_index
# Update mdata obs index
mdata.obs.index = new_mdata_index

print("\nAfter transformation:")
print(f"mdata.obs.index sample (first 3): {list(mdata.obs.index[:3])}")
print(f"tcell_metadata.index sample (first 3): {list(tcell_metadata.index[:3])}")
print(f"Matching cells: {mdata.obs.index.isin(tcell_metadata.index).sum()}/{len(mdata.obs)}")

Before transformation:
mdata.obs.index sample (first 3): ['LT1_AAACCTGAGGATATAC-1', 'LT1_AAACCTGAGTTACCCA-1', 'LT1_AAACCTGCAACACCCG-1']

After transformation:
mdata.obs.index sample (first 3): ['LT1_AAACCTGAGGATATAC-1', 'LT1_AAACCTGAGTTACCCA-1', 'LT1_AAACCTGCAACACCCG-1']
tcell_metadata.index sample (first 3): ['LT1_AAACCTGAGGATATAC-1', 'LT1_AAACCTGAGTTACCCA-1', 'LT1_AAACCTGCAACACCCG-1']
Matching cells: 103742/111062


In [21]:
all_metadata = pd.read_csv("GSE139555_all_metadata.txt", sep="\t", index_col=0)
all_metadata

,UMAP_1,UMAP_2,ident,patient,sample,source,clonotype
LT1_AAACCTGAGGATATAC-1,2.102660,-2.834420,7,Lung1,LT1,Tumor,lung1.tn.C1
LT1_AAACCTGAGTTACCCA-1,5.154273,-1.666514,5,Lung1,LT1,Tumor,lung1.tn.C3
LT1_AAACCTGCAACACCCG-1,2.792480,1.432129,12,Lung1,LT1,Tumor,lung1.tn.C5
LT1_AAACCTGCATCTCCCA-1,3.219543,2.568602,1,Lung1,LT1,Tumor,lung1.tn.C8
LT1_AAACCTGGTTCGTCTC-1,2.544952,4.659250,21,Lung1,LT1,Tumor,lung1.tn.C12
...,...,...,...,...,...,...,...
RB3_TTTGTCACACAGGCCT-1,5.002127,-2.034373,5,Renal3,RB3,Blood,renal3.tnb.C38
RB3_TTTGTCACACCCATTC-1,2.711555,-6.873210,6,Renal3,RB3,Blood,NaN
RB3_TTTGTCACACGCTTTC-1,2.839453,1.408865,1,Renal3,RB3,Blood,NaN
RB3_TTTGTCATCACGACTA-1,1.347771,-5.012864,6,Renal3,RB3,Blood,renal3.tnb.C198


In [22]:
# Create barcode mapping (handle concatenation suffixes)
mdata_base_to_full = {bc.rsplit('-', 1)[0] if bc.count('-') >= 2 else bc: bc for bc in mdata.obs_names}
mdata_base_to_full = {k: v for k, v in mdata_base_to_full.items() if k not in mdata_base_to_full or mdata_base_to_full[k] == v}


In [23]:
# Add 'isT' column: True if cell is in tcell_metadata, False otherwise
tcell_set = set(tcell_metadata.index)
mdata.obs['isT'] = mdata.obs.index.isin(tcell_set)

print(f"T cells (isT=True): {mdata.obs['isT'].sum()}")
print(f"Non-T cells (isT=False): {(~mdata.obs['isT']).sum()}")

# Initialize metadata columns
mdata.obs['ident'] = None
mdata.obs['clone_loc'] = None

# For T cells: take metadata from tcell_metadata
tcell_matched = 0
for meta_bc in tcell_metadata.index:
    mdata_bc = (meta_bc if meta_bc in mdata.obs_names else 
               mdata_base_to_full.get(meta_bc) or 
               (mdata_base_to_full.get(meta_bc.rsplit('-', 1)[0]) if meta_bc.count('-') >= 2 else None))
    if mdata_bc:
        mdata.obs.loc[mdata_bc, 'ident'] = tcell_metadata.loc[meta_bc, 'ident']

        # Parse clonotype like "lung1.tn.C1" and keep the middle segment as clone_loc ("tn")
        clonotype_val = tcell_metadata.loc[meta_bc, 'clonotype'] if 'clonotype' in tcell_metadata.columns else None
        if pd.notna(clonotype_val):
            parts = str(clonotype_val).split('.')
            if len(parts) >= 3:
                mdata.obs.loc[mdata_bc, 'clone_loc'] = parts[1]

        # Also copy patient and source from tcell_metadata
        for col in ['patient', 'source']:
            if col in tcell_metadata.columns:
                if col not in mdata.obs.columns:
                    mdata.obs[col] = None
                mdata.obs.loc[mdata_bc, col] = tcell_metadata.loc[meta_bc, col]
        tcell_matched += 1

# For non-T cells: take ident from all_metadata
all_meta_set = set(all_metadata.index)
nontcell_matched = 0
for idx in mdata.obs.index[~mdata.obs['isT']]:
    # Try to find matching barcode in all_metadata
    meta_bc = (idx if idx in all_meta_set else 
              (idx.rsplit('-', 1)[0] if idx.count('-') >= 1 and idx.rsplit('-', 1)[0] in all_meta_set else None))
    if meta_bc and 'ident' in all_metadata.columns:
        mdata.obs.loc[idx, 'ident'] = str(all_metadata.loc[meta_bc, 'ident'])
        nontcell_matched += 1

print(f"\nT cells matched with tcell_metadata: {tcell_matched}")
print(f"Non-T cells matched with all_metadata: {nontcell_matched}")
print(f"Added columns: isT, ident, patient, source, clone_loc")

T cells (isT=True): 103742
Non-T cells (isT=False): 7320

T cells matched with tcell_metadata: 103742
Non-T cells matched with all_metadata: 7320
Added columns: isT, ident, patient, source, clone_loc


In [24]:
mdata.obs['ident']

LT1_AAACCTGAGGATATAC-1     8.3a-Trm
LT1_AAACCTGAGTTACCCA-1     4.3-TCF7
LT1_AAACCTGCAACACCCG-1      4.4-FOS
LT1_AAACCTGCATCTCCCA-1      4.4-FOS
LT1_AAACCTGGTTCGTCTC-1     4.3-TCF7
                            ...    
RB3_TTTGGTTCAAGGTTTC-1     8.3c-Trm
RB3_TTTGGTTCATCAGTCA-1     8.1-Teff
RB3_TTTGGTTTCAACACCA-1    4.6a-Treg
RB3_TTTGTCACACAGGCCT-1      8.2-Tem
RB3_TTTGTCATCACGACTA-1     8.3b-Trm
Name: ident, Length: 111062, dtype: object

In [25]:
import re

def parse_ident(ident_str):
    """Parse ident string to extract type and subtype. Only keep CD4, CD8, others."""
    if pd.isna(ident_str) or ident_str is None or ident_str == 'None':
        return None, None
    
    ident_str = str(ident_str)
    
    # Extract first digit and only keep CD4, CD8, classify others as 'others'
    first_digit_match = re.match(r'^(\d+)', ident_str)
    if first_digit_match:
        first_digit = int(first_digit_match.group(1))
        if first_digit == 4:
            cell_type = 'CD4'
        elif first_digit == 8:
            cell_type = 'CD8'
        else:
            cell_type = 'others'
    else:
        cell_type = 'others'
    
    # Extract subtype (string after '-')
    if '-' in ident_str:
        subtype = ident_str.split('-', 1)[1]
    else:
        subtype = None

    # Exhausted Trm gradations from ident prefix (paper 8.3a/b/c)
    if ident_str.startswith("8.3a"):
        subtype = "Trm_exh_L"
    elif ident_str.startswith("8.3b"):
        subtype = "Trm_exh_M"
    elif ident_str.startswith("8.3c"):
        subtype = "Trm_exh_H"

    return cell_type, subtype

# Apply parsing to all cells
type_list = []
subtype_list = []

for ident_val in mdata.obs['ident']:
    cell_type, subtype = parse_ident(ident_val)
    type_list.append(cell_type)
    subtype_list.append(subtype)


In [26]:

# Add new columns
mdata.obs['type'] = type_list
mdata.obs['subtype'] = subtype_list

# Convert to categorical for efficiency
mdata.obs['type'] = mdata.obs['type'].astype('category')
mdata.obs['subtype'] = mdata.obs['subtype'].astype('category')


In [27]:
# Print summary
print(f"Type distribution:")
print(mdata.obs['type'].value_counts())
print(f"\nSubtype distribution (top 10):")
print(mdata.obs['subtype'].value_counts())

Type distribution:
type
CD8       49269
CD4       49182
others    12611
Name: count, dtype: int64

Subtype distribution (top 10):
subtype
TCF7         12973
Tem          12367
FOS          11165
Trm_exh_L    10051
Teff          9064
Trm           8376
Treg          7808
Trm_exh_H     7145
Trm_exh_M     6554
MT            5312
RPL32         4566
IL6ST         4233
Mitosis       1371
KLRB1         1370
Chrom         1347
Name: count, dtype: int64


In [28]:
mdata.obs['patient'].value_counts()

patient
Lung2     14663
Lung3     12932
Lung6     12846
Lung1     11234
Endo1     10786
Lung4      9431
Renal2     7406
Lung5      5213
Endo2      4212
Endo3      4088
Renal1     4073
Colon1     3190
Renal3     3137
Colon2      531
Name: count, dtype: int64

In [32]:
mdata.obs['clone_loc'].value_counts()

clone_loc
tn     69688
tnb    26417
Name: count, dtype: int64

In [30]:
mdata.write("GSE139555_Tcells.h5mu")

e:\Anaconda\envs\bio\Lib\site-packages\mudata\_core\mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
e:\Anaconda\envs\bio\Lib\site-packages\mudata\_core\mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)
